In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.exc import OperationalError
import ast

### read parquet

In [2]:
parquet_file = "recipes.parquet"
#chunks = pd.read_parquet(parquet_file, engine="pyarrow", chunksize=chunk_size)
df = pd.read_parquet(parquet_file, engine="pyarrow")

In [3]:
df

,ID,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Carbohydrate,Label,Category
0,0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{'name': 'vegetable or chicken stock', 'quant...",30.0,426.0,7.0,30.0,NaN,vegetarian,garlish
1,1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{'name': 'whipping cream', 'quantity': 5.5, '...",177.0,403.0,23.0,18.0,NaN,meat,main dish
2,4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{'name': 'spinach souffle', 'quantity': 1.0, ...",55.0,547.0,32.0,20.0,NaN,vegetarian,main dish
3,5,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{'name': 'basil leaves', 'quantity': 10.5, 'u...",8.0,948.0,79.0,19.0,NaN,meat,main dish
4,6,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{'name': 'red-skinned potatoes', 'quantity': ...",10.0,NaN,NaN,NaN,NaN,meat,main dish
...,...,...,...,...,...,...,...,...,...,...,...
1071004,1174475,Maple Apple Baked Beans,1. Place beans in soup kettle; add water to co...,"[{'name': 'navy beans', 'quantity': 91, 'unit'...",375.0,518.1,19.0,17.3,72.3,meat,garlish
1071005,1174476,Blackberry Orange Scones,1. Sift about 2 cups of flour onto a piece of ...,"[{'name': 'flour', 'quantity': 91, 'unit': 'cu...",27.0,244.8,9.1,5.1,35.4,"pescetarian, vegetarian",dessert
1071006,1174477,Slow Cooker Garlic Chicken With Rosemary,"1. Place rosemary springs, 1 lemon half, celer...","[{'name': 'roasting chickens', 'quantity': 91,...",440.0,566.2,38.9,43.2,9.3,meat,main dish
1071007,1174478,Kapusta ( Cabbage and Kielbasa ),1. Saute bacon in large pan until browned. Le...,"[{'name': 'cabbage', 'quantity': 91, 'unit': '...",NaN,688.0,48.5,25.6,39.2,meat,main dish


In [4]:
df.columns

Index(['ID', 'Name', 'Instructions', 'Ingredients', 'Total Time', 'Calories',
       'Fat', 'Protein', 'Carbohydrate', 'Label', 'Category'],
      dtype='object')

In [5]:
df_upload_recipe = df

### database connection

In [6]:
from sqlalchemy import create_engine, text

# Replace these with actual values
DB_HOST = "frs-db.ch88c4s48jz3.eu-north-1.rds.amazonaws.com"
DB_NAME = "FRS"
DB_USER = "postgres"
DB_PASSWORD = "DefinedAtTheDisco01*"  
DB_PORT = "5432" 

# Create a database connection
engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'"))
        tables = [row[0] for row in result.fetchall()]
        
        print("✅ Connected to the database successfully!")
        print("Available tables:", tables)

except Exception as e:
    print("❌ Failed to connect to the database.")
    print("Error:", e)


✅ Connected to the database successfully!
Available tables: ['recipe_ingr', 'category', 'recipe', 'pref_recipe', 'preference', 'liked_recipes', 'users', 'saved_recipes', 'user_pref', 'allergy', 'ingredient']


## uplaod tables

In [7]:
chunk_size = 50000

In [8]:
def upload_table(df_to_upload, table):
    try:
        # 🔹 Insert data in chunks with proper rollback handling
        for i in range(0, len(df_to_upload), chunk_size):
            try:
                with engine.begin() as connection:  # Ensures transactions are handled safely
                    df_to_upload.iloc[i:i+chunk_size].to_sql(table, connection, if_exists="append", index=False)
                    print(f"Uploaded {i+chunk_size}/{len(df_to_upload)} rows")

            except OperationalError as e:
                print(f"Connection lost during batch {i}-{i+chunk_size}, retrying...")
                continue  # Retry the next batch

        print("Full recipe dataset uploaded successfully!")

    except Exception as e:
        print("Fatal error uploading data:", e)

### recipe (without category)

In [ ]:
# 🔹 Mapping DataFrame columns to Database Table columns
column_mapping = {
    "ID": "recipe_id",
    "Name": "recipe_name",
    "Instructions": "instruction",
    "Ingredients": "ingredient",  
    "Total Time": "total_time",
    "Calories": "calories",
    "Fat": "fat",
    "Protein": "protein",
    "Carbohydrate": "carb",
}


In [ ]:
df_upload_recipe = df_upload_recipe.rename(columns=column_mapping)
df_upload_recipe = df_upload_recipe[list(column_mapping.values())] # Select only the mapped columns

In [ ]:
df_upload_recipe.columns

Index(['recipe_id', 'recipe_name', 'instruction', 'ingredient', 'total_time',
       'calories', 'fat', 'protein', 'carb'],
      dtype='object')

In [ ]:
try:
    # 🔹 Insert data in chunks with proper rollback handling
    for i in range(0, len(df_upload_recipe), chunk_size):
        try:
            with engine.begin() as connection:  # Ensures transactions are handled safely
                df_upload_recipe.iloc[i:i+chunk_size].to_sql("recipe", connection, if_exists="append", index=False)
                print(f"Uploaded {i+chunk_size}/{len(df_upload_recipe)} rows")

        except OperationalError as e:
            print(f"Connection lost during batch {i}-{i+chunk_size}, retrying...")
            continue  # Retry the next batch

    print("Full recipe dataset uploaded successfully!")

except Exception as e:
    print("Fatal error uploading data:", e)

✅ Uploaded 50000/1071009 rows
✅ Uploaded 100000/1071009 rows
✅ Uploaded 150000/1071009 rows
✅ Uploaded 200000/1071009 rows
✅ Uploaded 250000/1071009 rows
✅ Uploaded 300000/1071009 rows
✅ Uploaded 350000/1071009 rows
✅ Uploaded 400000/1071009 rows
✅ Uploaded 450000/1071009 rows
✅ Uploaded 500000/1071009 rows
✅ Uploaded 550000/1071009 rows
✅ Uploaded 600000/1071009 rows
✅ Uploaded 650000/1071009 rows
✅ Uploaded 700000/1071009 rows
✅ Uploaded 750000/1071009 rows
✅ Uploaded 800000/1071009 rows
✅ Uploaded 850000/1071009 rows
✅ Uploaded 900000/1071009 rows
✅ Uploaded 950000/1071009 rows
✅ Uploaded 1000000/1071009 rows
✅ Uploaded 1050000/1071009 rows
✅ Uploaded 1100000/1071009 rows
🚀 Full recipe dataset uploaded successfully!


### ingredient

In [ ]:
df.columns

Index(['ID', 'Name', 'Instructions', 'Ingredients', 'Total Time', 'Calories',
       'Fat', 'Protein', 'Carbohydrate', 'Label', 'Category'],
      dtype='object')

In [ ]:
# 🔹 Extract ingredient names from the list
ingredient_set = set()  # To store unique ingredient names

for ingredients in df["Ingredients"].dropna():  # Ignore missing values
    ingredient_list = ast.literal_eval(ingredients)  # Convert string to list
    for item in ingredient_list:
        ingredient_name = item["name"].strip().replace("'", "''")  # Escape apostrophes
        ingredient_set.add(ingredient_name)

# 🔹 Convert to DataFrame
ingredient_df = pd.DataFrame({"ingr_name": list(ingredient_set)})

In [ ]:
ingredient_df

,ingr_name
0,canfruit cocktail
1,red cinnamon candies
2,good quality matcha powder
3,millet or white quinoa
4,brown sugar-packed
...,...
186896,pizza dough yeast
186897,cauliflower or broccoli floret
186898,vegetable stock or chicken stock or broth
186899,instant mint-chocolate pudding mix


In [ ]:
try:
    # 🔹 Insert ingredients in chunks
    for i in range(0, len(ingredient_df), chunk_size):
        ingredient_df.iloc[i:i+chunk_size].to_sql("ingredient", engine, if_exists="append", index=False)
        print(f"Uploaded {i+chunk_size}/{len(ingredient_df)} ingredients")

    print("Successfully uploaded all unique ingredients!")

except Exception as e:
    print("Error uploading ingredients:", e)

Uploaded 50000/186901 ingredients
Uploaded 100000/186901 ingredients
Uploaded 150000/186901 ingredients
Uploaded 200000/186901 ingredients
Successfully uploaded all unique ingredients!


### category

In [ ]:
df_upload_recipe = df

In [ ]:
# 🔹 Mapping DataFrame columns to Database Table columns
column_mapping = {
    "Category": "cat_name"
}

df_upload_recipe = df_upload_recipe.rename(columns=column_mapping)
df_upload_recipe = df_upload_recipe[list(column_mapping.values())] # Select only the mapped columns

In [ ]:
df_upload_recipe

,cat_name
0,garlish
1,main dish
2,main dish
3,main dish
4,main dish
...,...
1071004,garlish
1071005,dessert
1071006,main dish
1071007,main dish


In [ ]:
unique_categories = df_upload_recipe['cat_name'].dropna().unique()
print(unique_categories)

['garlish' 'main dish' 'dessert' 'appetizer' 'beverage' 'soup' 'breakfast'
 'bread']


In [ ]:
df_upload_recipe = pd.DataFrame({"cat_name": unique_categories})


In [ ]:
df_upload_recipe

,cat_name
0,garlish
1,main dish
2,dessert
3,appetizer
4,beverage
5,soup
6,breakfast
7,bread


In [ ]:
category_data = [
    (0, "garlish"),
    (1, "main dish"),
    (2, "dessert"),
    (3, "appetizer"),
    (4, "beverage"),
    (5, "soup"),
    (6, "breakfast"),
    (7, "bread")
]

# 🔹 Convert to DataFrame
category_df = pd.DataFrame(category_data, columns=["category_id", "cat_name"])


In [ ]:
category_df

,category_id,cat_name
0,0,garlish
1,1,main dish
2,2,dessert
3,3,appetizer
4,4,beverage
5,5,soup
6,6,breakfast
7,7,bread


In [ ]:
upload_table(category_df, "category")

Connection lost during batch 0-50000, retrying...
Full recipe dataset uploaded successfully!


### pref

In [ ]:
df.columns

Index(['ID', 'Name', 'Instructions', 'Ingredients', 'Total Time', 'Calories',
       'Fat', 'Protein', 'Carbohydrate', 'Label', 'Category'],
      dtype='object')

In [ ]:
df_upload_recipe = df

In [ ]:
# 🔹 Mapping DataFrame columns to Database Table columns
column_mapping = {
    "Label": "pref_name"
}

df_upload_recipe = df_upload_recipe.rename(columns=column_mapping)
df_upload_recipe = df_upload_recipe[list(column_mapping.values())] # Select only the mapped columns

In [ ]:
unique_categories = df_upload_recipe['pref_name'].dropna().unique()
print(unique_categories)

['vegetarian' 'meat' 'dairy-free, gluten-free, pescetarian, vegetarian'
 'pescetarian' 'pescetarian, vegetarian'
 'dairy-free, gluten-free, pescetarian, vegan, vegetarian'
 'gluten-free, vegetarian' 'dairy-free, gluten-free, pescetarian']


In [ ]:
df_upload_recipe = pd.DataFrame({"pref_name": unique_categories})

In [ ]:
['vegetarian' 'meat' 'dairy-free, gluten-free, pescetarian, vegetarian'
 'pescetarian' 'pescetarian, vegetarian'
 'dairy-free, gluten-free, pescetarian, vegan, vegetarian'
 'gluten-free, vegetarian' 'dairy-free, gluten-free, pescetarian']

['vegetarianmeatdairy-free, gluten-free, pescetarian, vegetarianpescetarianpescetarian, vegetariandairy-free, gluten-free, pescetarian, vegan, vegetariangluten-free, vegetariandairy-free, gluten-free, pescetarian']

In [ ]:
label_data = [
    (0, "vegetarian"),
    (1, "meat"),
    (2, "dairy-free"),
    (3, "pescetarian"),
    (4, "gluten-free"),
    (5, "vegan"),
]

# 🔹 Convert to DataFrame
label_df = pd.DataFrame(label_data, columns=["pref_id", "pref_name"])

In [ ]:
label_df

,pref_id,pref_name
0,0,vegetarian
1,1,meat
2,2,dairy-free
3,3,pescetarian
4,4,gluten-free
5,5,vegan


In [ ]:
upload_table(label_df, "preference")

Uploaded 50000/6 rows
Full recipe dataset uploaded successfully!


### pref_recipe

In [ ]:
df

,ID,Name,Instructions,Ingredients,Total Time,Calories,Fat,Protein,Carbohydrate,Label,Category
0,0,"Lentil, Apple, and Turkey Wrap","1. Place the stock, lentils, celery, carrot, t...","[{'name': 'vegetable or chicken stock', 'quant...",30.0,426.0,7.0,30.0,NaN,vegetarian,garlish
1,1,Boudin Blanc Terrine with Red Onion Confit,Combine first 9 ingredients in heavy saucepan...,"[{'name': 'whipping cream', 'quantity': 5.5, '...",177.0,403.0,23.0,18.0,NaN,meat,main dish
2,4,Spinach Noodle Casserole,Preheat oven to 350°F. Lightly grease 8x8x2-in...,"[{'name': 'spinach souffle', 'quantity': 1.0, ...",55.0,547.0,32.0,20.0,NaN,vegetarian,main dish
3,5,The Best Blts,"Mix basil, mayonnaise and butter in processor ...","[{'name': 'basil leaves', 'quantity': 10.5, 'u...",8.0,948.0,79.0,19.0,NaN,meat,main dish
4,6,Ham and Spring Vegetable Salad with Shallot Vi...,Cook potatoes and carrots in pot of boiling s...,"[{'name': 'red-skinned potatoes', 'quantity': ...",10.0,NaN,NaN,NaN,NaN,meat,main dish
...,...,...,...,...,...,...,...,...,...,...,...
1071004,1174475,Maple Apple Baked Beans,1. Place beans in soup kettle; add water to co...,"[{'name': 'navy beans', 'quantity': 91, 'unit'...",375.0,518.1,19.0,17.3,72.3,meat,garlish
1071005,1174476,Blackberry Orange Scones,1. Sift about 2 cups of flour onto a piece of ...,"[{'name': 'flour', 'quantity': 91, 'unit': 'cu...",27.0,244.8,9.1,5.1,35.4,"pescetarian, vegetarian",dessert
1071006,1174477,Slow Cooker Garlic Chicken With Rosemary,"1. Place rosemary springs, 1 lemon half, celer...","[{'name': 'roasting chickens', 'quantity': 91,...",440.0,566.2,38.9,43.2,9.3,meat,main dish
1071007,1174478,Kapusta ( Cabbage and Kielbasa ),1. Saute bacon in large pan until browned. Le...,"[{'name': 'cabbage', 'quantity': 91, 'unit': '...",NaN,688.0,48.5,25.6,39.2,meat,main dish


In [ ]:
label_data = [
    (0, "vegetarian"),
    (1, "meat"),
    (2, "dairy-free"),
    (3, "pescetarian"),
    (4, "gluten-free"),
    (5, "vegan"),
]

In [ ]:
def generate_pref_recipe(df, label_data):
    # Convert label_data into a dictionary {label_name: pref_id}
    label_map = {label: pref_id for pref_id, label in label_data}
    #print(label_map)
    pref_recipe_data = []
    
    # Iterate through each recipe
    for _, row in df.iterrows():
        recipe_id = row["ID"]
        labels = row["Label"]
        
        if pd.isna(labels):  # Handle missing labels
            continue

        labels = labels.split(",")  # Split multi-label strings
        #print(labels)
        # Map labels to pref_id
        for label in labels:
            label = label.strip()  # Remove spaces
            if label in label_map:
                pref_recipe_data.append((recipe_id, label_map[label]))

    return pref_recipe_data

In [ ]:
pref_recipe = generate_pref_recipe(df, label_data)
#print(pref_recipe)

KeyboardInterrupt: 

In [ ]:
pref_recipe

[(0, 0),
 (1, 1),
 (4, 0),
 (5, 1),
 (6, 1),
 (7, 0),
 (9, 1),
 (10, 1),
 (11, 1),
 (13, 1),
 (14, 2),
 (14, 4),
 (14, 3),
 (14, 0),
 (15, 3),
 (16, 0),
 (17, 1),
 (18, 3),
 (19, 1),
 (21, 1),
 (22, 3),
 (24, 3),
 (25, 0),
 (26, 0),
 (27, 2),
 (27, 4),
 (27, 3),
 (27, 0),
 (28, 3),
 (28, 0),
 (29, 3),
 (29, 0),
 (30, 3),
 (30, 0),
 (31, 2),
 (31, 4),
 (31, 3),
 (31, 5),
 (31, 0),
 (33, 1),
 (34, 0),
 (35, 1),
 (36, 3),
 (36, 0),
 (37, 2),
 (37, 4),
 (37, 3),
 (37, 5),
 (37, 0),
 (38, 3),
 (40, 0),
 (41, 3),
 (42, 3),
 (42, 0),
 (43, 3),
 (44, 1),
 (45, 2),
 (45, 4),
 (45, 3),
 (45, 5),
 (45, 0),
 (46, 1),
 (47, 1),
 (48, 0),
 (49, 2),
 (49, 4),
 (49, 3),
 (49, 0),
 (50, 1),
 (51, 0),
 (54, 0),
 (55, 1),
 (56, 1),
 (57, 0),
 (58, 3),
 (58, 0),
 (59, 0),
 (60, 0),
 (61, 1),
 (62, 0),
 (63, 0),
 (64, 0),
 (65, 0),
 (66, 3),
 (66, 0),
 (68, 0),
 (69, 1),
 (70, 0),
 (71, 1),
 (72, 0),
 (73, 2),
 (73, 4),
 (73, 3),
 (73, 0),
 (74, 0),
 (75, 0),
 (76, 2),
 (76, 4),
 (76, 3),
 (76, 5),
 (76, 0

In [ ]:
df_pref_recipe = pd.DataFrame(pref_recipe, columns=["recipe_id", "pref_id"])

In [ ]:
df_pref_recipe

,recipe_id,pref_id
0,0,0
1,1,1
2,4,0
3,5,1
4,6,1
...,...,...
1648194,1174476,3
1648195,1174476,0
1648196,1174477,1
1648197,1174478,1


In [ ]:
upload_table(df_pref_recipe, "pref_recipe")

Connection lost during batch 0-50000, retrying...
Uploaded 100000/1648199 rows
Uploaded 150000/1648199 rows
Uploaded 200000/1648199 rows
Uploaded 250000/1648199 rows
Uploaded 300000/1648199 rows
Uploaded 350000/1648199 rows
Uploaded 400000/1648199 rows
Uploaded 450000/1648199 rows
Uploaded 500000/1648199 rows
Uploaded 550000/1648199 rows
Uploaded 600000/1648199 rows
Uploaded 650000/1648199 rows
Uploaded 700000/1648199 rows
Uploaded 750000/1648199 rows
Uploaded 800000/1648199 rows
Uploaded 850000/1648199 rows
Uploaded 900000/1648199 rows
Uploaded 950000/1648199 rows
Uploaded 1000000/1648199 rows
Uploaded 1050000/1648199 rows
Uploaded 1100000/1648199 rows
Uploaded 1150000/1648199 rows
Uploaded 1200000/1648199 rows
Uploaded 1250000/1648199 rows
Uploaded 1300000/1648199 rows
Uploaded 1350000/1648199 rows
Uploaded 1400000/1648199 rows
Uploaded 1450000/1648199 rows
Uploaded 1500000/1648199 rows
Uploaded 1550000/1648199 rows
Uploaded 1600000/1648199 rows
Uploaded 1650000/1648199 rows
Full rec

### recipe_ingr

In [ ]:
# recipe_id, ingr_id, quantity, unit

In [9]:
df.columns

Index(['ID', 'Name', 'Instructions', 'Ingredients', 'Total Time', 'Calories',
       'Fat', 'Protein', 'Carbohydrate', 'Label', 'Category'],
      dtype='object')

In [10]:
df['Ingredients'][0]

"[{'name': 'vegetable or chicken stock', 'quantity': 4.0, 'unit': 'cups'}, {'name': 'brown lentils', 'quantity': 1.0, 'unit': 'cup'}, {'name': 'french green lentils', 'quantity': 0.5, 'unit': 'cup'}, {'name': 'celery', 'quantity': 2.0, 'unit': 'stalks'}, {'name': 'carrot', 'quantity': 1.0, 'unit': None}, {'name': 'thyme', 'quantity': 1.0, 'unit': 'sprig'}, {'name': 'kosher salt', 'quantity': 1.0, 'unit': 'teaspoon'}, {'name': 'tomato', 'quantity': 1.0, 'unit': None}, {'name': 'fuji apple', 'quantity': 1.0, 'unit': None}, {'name': 'lemon juice', 'quantity': 1.0, 'unit': 'tablespoon'}, {'name': 'extra-virgin olive oil', 'quantity': 2.0, 'unit': 'teaspoons'}, {'name': 'black pepper', 'quantity': None, 'unit': None}, {'name': 'whole-wheat lavash or flour tortillas', 'quantity': 3.0, 'unit': 'sheets'}, {'name': 'turkey breast', 'quantity': 0.75, 'unit': 'pound'}, {'name': 'bibb lettuce', 'quantity': 0.5, 'unit': 'head'}]"

In [11]:
query_ingredient = "SELECT ingr_id, ingr_name FROM ingredient;"
ingredient_df = pd.read_sql(query_ingredient, engine)

In [12]:
ingredient_df

,ingr_id,ingr_name
0,1,canfruit cocktail
1,2,red cinnamon candies
2,3,good quality matcha powder
3,4,millet or white quinoa
4,5,brown sugar-packed
...,...,...
186896,186897,pizza dough yeast
186897,186898,cauliflower or broccoli floret
186898,186899,vegetable stock or chicken stock or broth
186899,186900,instant mint-chocolate pudding mix


In [13]:
def normalize_ingredient_name(name):
    return name.strip().replace("'", "''").lower()

In [14]:
ingredient_map = {row["ingr_name"].strip().lower(): row["ingr_id"] for _, row in ingredient_df.iterrows()}

In [15]:
ingredient_map

{'canfruit cocktail': 1,
 'red cinnamon candies': 2,
 'good quality matcha powder': 3,
 'millet or white quinoa': 4,
 'brown sugar-packed': 5,
 'basic halvah': 6,
 'cooked artichoke heart': 7,
 'lemon grated lemon': 8,
 'organic chicken': 9,
 'ylang ylang': 10,
 'white vinegar and water': 11,
 'triple cereal': 12,
 'soy sauce or soy sauce': 13,
 'tangy onion paratha': 14,
 'breadcrumb or panko breadcrumb': 15,
 'flour butter mixed': 16,
 'celery salt or seasoning salt': 17,
 'date bread mix': 18,
 'shake worcestershire sauce': 19,
 'fish or shrimp or clam': 20,
 'indio marionberry vodka': 21,
 "clark''s barbecue sauce": 22,
 'vanilla mousse': 23,
 'ramen or instant noodle or noodle': 24,
 'harvest or sweet onion': 25,
 'strawberry-vin santo sauce': 26,
 'blood orange italian soda': 27,
 'rosenborg beelablu cheese': 28,
 'sunflower seed or kernal': 29,
 'tomato sauce or tomato mushroom sauce': 30,
 'active yeast or bread machine yeast': 31,
 'wormwood bitter': 32,
 'monterey jack cheese

In [16]:
def generate_ingr_recipe(df):
    #print(label_map)
    recipe_ingr_data = []
    # Iterate through each recipe
    for _, row in df.iterrows():
        recipe_id = row["ID"]
        ingredients = row["Ingredients"]
        
        if pd.isna(ingredients):  # Handle missing labels
            continue

        ingredients_list = ast.literal_eval(ingredients)  # Convert string to list

        for item in ingredients_list:
            ingredient_name = normalize_ingredient_name(item["name"])
            quantity = item.get("quantity", None)  # Get quantity, default to None
            unit = item.get("unit", None)  # Get unit, default to None

            # 🔹 Find ingr_id from the real database mapping
            ingr_id = ingredient_map.get(ingredient_name)

            if ingr_id:
                recipe_ingr_data.append((recipe_id, ingr_id, quantity, unit))
            else:
                print(f"⚠️ Warning: Ingredient '{ingredient_name}' still not found!")

    return recipe_ingr_data

In [17]:
missing_ingredients = set()
for ingredients in df["Ingredients"].dropna():
    ingredients_list = ast.literal_eval(ingredients)  # Convert string to list

    for item in ingredients_list:
        ingredient_name = normalize_ingredient_name(item["name"])

        if ingredient_name not in ingredient_map:
            missing_ingredients.add(ingredient_name)

# 🔹 Print missing ingredients
print("⚠️ Missing ingredients:", missing_ingredients)


⚠️ Missing ingredients: set()


In [18]:
len(missing_ingredients)
# 3694

0

In [19]:
recipe_ingr_data = generate_ingr_recipe(df)

In [20]:
df_recipe_ingr = pd.DataFrame(recipe_ingr_data, columns=["recipe_id", "ingr_id", "quantity", "unit"])

In [21]:
df_recipe_ingr

,recipe_id,ingr_id,quantity,unit
0,0,90699,4.0,cups
1,0,155455,1.0,cup
2,0,108948,0.5,cup
3,0,113066,2.0,stalks
4,0,18365,1.0,None
...,...,...,...,...
9446508,1174479,170574,34,teaspoon
9446509,1174479,11856,44,teaspoon
9446510,1174479,109775,34,None
9446511,1174479,126876,49,None


In [22]:
"""
Fatal error uploading data: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "recipe_ingr_pkey"
DETAIL:  Key (recipe_id, ingr_id)=(1, 65614) already exists."""

print(df_recipe_ingr[(df_recipe_ingr["recipe_id"] == 1) & (df_recipe_ingr["ingr_id"] == 65614)])


    recipe_id  ingr_id quantity  unit
18          1    65614      3.0  None
34          1    65614     None  None


In [23]:
df_recipe_ingr = df_recipe_ingr.drop_duplicates(subset=["recipe_id", "ingr_id"], keep="first")

In [24]:
df_recipe_ingr

,recipe_id,ingr_id,quantity,unit
0,0,90699,4.0,cups
1,0,155455,1.0,cup
2,0,108948,0.5,cup
3,0,113066,2.0,stalks
4,0,18365,1.0,None
...,...,...,...,...
9446508,1174479,170574,34,teaspoon
9446509,1174479,11856,44,teaspoon
9446510,1174479,109775,34,None
9446511,1174479,126876,49,None


In [ ]:
import re

def clean_quantity(value):
    """
    Converts quantity values to valid float numbers.
    - Handles ranges like '10-15' → converts to 10.0 (lower bound).
    - Handles fractions like '1/2' → converts to 0.5.
    - Returns None if conversion is not possible.
    """
    if pd.isna(value) or value is None:
        return None  # Keep null values
    
    value = str(value).strip()  # Ensure it's a string
    
    # Handle ranges like "10-15" → take the first number
    if "-" in value:
        value = value.split("-")[0]

    # Convert fractions like "1/2" to float
    if "/" in value:
        try:
            return float(value.split("/")[0]) / float(value.split("/")[1])
        except:
            return None  # If conversion fails, return None
    
    # Convert normal numbers
    try:
        return float(value)
    except ValueError:
        return None  # If not a valid number, return None

# 🔹 Apply the cleaning function to `quantity` column
df_recipe_ingr["quantity"] = df_recipe_ingr["quantity"].apply(clean_quantity)

# 🔹 Print invalid quantities to debug
invalid_quantities = df_recipe_ingr[df_recipe_ingr["quantity"].isna()]
print("⚠️ Invalid Quantities:", invalid_quantities)


In [26]:
upload_table(df_recipe_ingr, "recipe_ingr")

Uploaded 50000/9121530 rows
Uploaded 100000/9121530 rows
Uploaded 150000/9121530 rows
Uploaded 200000/9121530 rows
Uploaded 250000/9121530 rows
Uploaded 300000/9121530 rows
Uploaded 350000/9121530 rows
Uploaded 400000/9121530 rows
Uploaded 450000/9121530 rows
Uploaded 500000/9121530 rows
Uploaded 550000/9121530 rows
Uploaded 600000/9121530 rows
Uploaded 650000/9121530 rows
Uploaded 700000/9121530 rows
Uploaded 750000/9121530 rows
Uploaded 800000/9121530 rows
Uploaded 850000/9121530 rows
Uploaded 900000/9121530 rows
Uploaded 950000/9121530 rows
Uploaded 1000000/9121530 rows
Uploaded 1050000/9121530 rows
Uploaded 1100000/9121530 rows
Uploaded 1150000/9121530 rows
Uploaded 1200000/9121530 rows
Uploaded 1250000/9121530 rows
Uploaded 1300000/9121530 rows
Uploaded 1350000/9121530 rows
Uploaded 1400000/9121530 rows
Uploaded 1450000/9121530 rows
Uploaded 1500000/9121530 rows
Uploaded 1550000/9121530 rows
Uploaded 1600000/9121530 rows
Uploaded 1650000/9121530 rows
Uploaded 1700000/9121530 rows


In [27]:
from sqlalchemy.exc import OperationalError, IntegrityError

def upload_table_skip_rows(df_to_upload, table, start_row=0, chunk_size=50000):
    """
    Uploads data in chunks, resuming from a specific row (start_row).
    - Skips syntax errors & continues to the next batch.
    - Automatically resumes from where it left off.
    """
    try:
        for i in range(start_row, len(df_to_upload), chunk_size):
            try:
                with engine.begin() as connection:
                    df_to_upload.iloc[i:i+chunk_size].to_sql(
                        table, connection, if_exists="append", index=False, method="multi"
                    )
                    print(f"✅ Uploaded {i+chunk_size}/{len(df_to_upload)} rows")

            except OperationalError as e:
                print(f"⚠️ Connection lost at batch {i}-{i+chunk_size}, retrying...")
                continue  # Retry next batch

            except IntegrityError as e:
                print(f"⚠️ Skipping batch {i}-{i+chunk_size} due to IntegrityError: {e}")
                continue  # Skip problematic batch

        print("🚀 Full dataset uploaded successfully!")

    except Exception as e:
        print("❌ Fatal error uploading data:", e)




In [28]:
# 🔹 Resume from row 6450000
upload_table_skip_rows(df_recipe_ingr, "recipe_ingr", start_row=6450000)

⚠️ Connection lost at batch 6450000-6500000, retrying...
✅ Uploaded 6550000/9121530 rows
✅ Uploaded 6600000/9121530 rows
✅ Uploaded 6650000/9121530 rows
✅ Uploaded 6700000/9121530 rows
✅ Uploaded 6750000/9121530 rows
✅ Uploaded 6800000/9121530 rows
✅ Uploaded 6850000/9121530 rows
✅ Uploaded 6900000/9121530 rows
✅ Uploaded 6950000/9121530 rows
✅ Uploaded 7000000/9121530 rows
✅ Uploaded 7050000/9121530 rows
✅ Uploaded 7100000/9121530 rows
✅ Uploaded 7150000/9121530 rows
✅ Uploaded 7200000/9121530 rows
✅ Uploaded 7250000/9121530 rows
✅ Uploaded 7300000/9121530 rows
✅ Uploaded 7350000/9121530 rows
✅ Uploaded 7400000/9121530 rows
✅ Uploaded 7450000/9121530 rows
✅ Uploaded 7500000/9121530 rows
✅ Uploaded 7550000/9121530 rows
✅ Uploaded 7600000/9121530 rows
✅ Uploaded 7650000/9121530 rows
✅ Uploaded 7700000/9121530 rows
✅ Uploaded 7750000/9121530 rows
✅ Uploaded 7800000/9121530 rows
✅ Uploaded 7850000/9121530 rows
✅ Uploaded 7900000/9121530 rows
✅ Uploaded 7950000/9121530 rows
✅ Uploaded 8000